# BadGraph Attack Document Generation

This notebook implements the BadGraph attack-document generation workflow described in the paper. It converts target-query information into neutral append-only documents that can be inserted before GraphRAG indexing.

## Conceptual Background

<p align="center">
  <img src="assets/graphrag_overview.png" alt="GraphRAG overview" width="760">
</p>

GraphRAG systems do not retrieve only raw text chunks. During indexing, they extract graph units such as entities, relationships, community summaries, and source chunks. During retrieval, a query reaches initial semantic anchors and then expands over graph structure before the final context is sent to the generator. BadGraph targets this graph-retrieval stage.

<p align="center">
  <img src="assets/concept.png" alt="BadGraph structural knowledge isolation concept" width="760">
</p>

BadGraph is a targeted availability attack. It appends neutral documents before indexing, so normal GraphRAG extraction turns them into additional entities, relationships, and source units. Its goal is to reduce evidence availability by adding competing topology around query-related anchors, rather than forcing a specific wrong answer.

## Notebook-to-Paper Mapping

<p align="center">
  <img src="assets/framework.png" alt="BadGraph attack workflow" width="760">
</p>

The notebook follows the three phases in the paper framework:

1. **Anchor prediction**: infer semantic anchors that GraphRAG retrieval is likely to reach for the target query.
2. **Adversarial subgraph construction**: allocate the append-only graph budget into the Trap Component and, for MS-GraphRAG community-summary retrieval, the Confusion Component implemented through bridge relations.
3. **Linker document generation**: write neutral archive/catalogue-style text that states the intended entity relationships clearly enough for normal GraphRAG indexing to extract them.

The output is a JSONL file containing generated documents and metadata for inspecting the intended structure.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import textwrap
import yaml

ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config.yaml"
GRAPH_DIR = ROOT / "graphs"
OUT_DIR = ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_PATH}. Run this notebook from the BadGraph artifact root.")

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

attack_cfg = config.get("attack", {})
BUDGET_RATIO = float(attack_cfg.get("budget_ratio", 0.005))
NUM_DISTRACTORS = int(attack_cfg.get("num_distractors", 3))
NUM_LEAVES = int(attack_cfg.get("num_leaves_per_distractor", 5))
BRIDGE_BUDGET_RATIO = float(attack_cfg.get("bridge_budget_ratio", 0.10))
BRIDGE_ANCHORS = list(attack_cfg.get("bridge_anchors", []))
GRAPH_STATS = {}

def clean_label(text, max_len=60):
    label = re.sub(r"\s+", " ", str(text).strip())
    return label[:max_len] or "Archive Entity"

NEUTRAL_PLACES = [
    "Northbridge", "Lakeview", "Riverside", "Westhaven", "Meridian", "Oakfield",
    "Silverton", "Brookside", "Eastgate", "Hillcrest", "Fairview", "Redwood",
]
NEUTRAL_DOMAINS = [
    "Heritage", "Civic", "Regional", "Cultural", "Museum", "Archive", "Bibliographic",
    "Reference", "Collection", "Gazetteer", "Chronology", "Attribution", "Affiliation",
    "Production", "Publication", "Credit", "Location", "Role", "Lineage", "Provenance",
]
NEUTRAL_FORMS = [
    "Register", "Catalogue", "Index", "Survey", "Ledger", "Directory", "Inventory",
    "Record", "Series", "Compendium", "Dossier", "Crosswalk", "Finding Aid",
    "Concordance", "Profile",
]

def stable_int(*parts):
    payload = "\n".join(str(p) for p in parts)
    return int(hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16], 16)

def neutral_entity_name(anchor, kind, i, j=None):
    # Neutral descriptor construction affects extraction/retrieval salience:
    # plausible catalogue-style names improve entity extraction and context competition over artificial placeholders.
    seed = stable_int("badgraph", anchor, kind, i, -1 if j is None else j)
    place = NEUTRAL_PLACES[seed % len(NEUTRAL_PLACES)]
    domain = NEUTRAL_DOMAINS[(seed // 13) % len(NEUTRAL_DOMAINS)]
    form = NEUTRAL_FORMS[(seed // 97) % len(NEUTRAL_FORMS)]
    series = 100 + (seed % 900)
    return f"{place} {domain} {form} Series {series}"

SUPPORTED_SYSTEMS = {"graphrag", "lightrag", "fastgraphrag"}

def normalize_system(system):
    value = str(system or "graphrag").strip().lower()
    if value not in SUPPORTED_SYSTEMS:
        raise ValueError(f"Unsupported system {system!r}. Use one of: {sorted(SUPPORTED_SYSTEMS)}")
    return value

def uses_bridge_component(system):
    return normalize_system(system) == "graphrag"

def graph_path_for_system(system):
    return GRAPH_DIR / normalize_system(system) / "extracted_graph.jsonl"

def graph_stats(system):
    system = normalize_system(system)
    if system in GRAPH_STATS:
        return GRAPH_STATS[system]
    path = graph_path_for_system(system)
    if not path.exists():
        raise FileNotFoundError(f"Missing graph file for {system}: {path}")
    edges = set()
    chunks = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            chunks += 1
            row = json.loads(line)
            for rel in row.get("relationships", []) or []:
                source = clean_label(rel.get("source"))
                target = clean_label(rel.get("target"))
                if source and target and source != target:
                    edges.add(tuple(sorted((source, target))))
    stats = {"chunks": chunks, "edges": len(edges)}
    GRAPH_STATS[system] = stats
    return stats

def structural_budget_for_case(case):
    system = normalize_system(case.get("system", "graphrag"))
    graph_edges = int(case.get("graph_edge_count") or graph_stats(system)["edges"])
    total = max(1, int(graph_edges * BUDGET_RATIO))
    if uses_bridge_component(system):
        bridge = max(1, int(total * BRIDGE_BUDGET_RATIO)) if total > 1 else 0
        bridge = min(bridge, total)
    else:
        bridge = 0
    return {
        "graph_edges": graph_edges,
        "budget_ratio": BUDGET_RATIO,
        "total": total,
        "trap": total - bridge,
        "bridge": bridge,
    }

print(f"Graph budget ratio: {BUDGET_RATIO:.3%}; Bridge share: {BRIDGE_BUDGET_RATIO:.0%}")


## 1. Configure target queries

Replace `TARGET_CASES` with the target query set being evaluated. Each case needs a query and target GraphRAG system name, and may optionally include a short `background` field with target-domain context for anchor discovery. The following cells predict anchor candidates, compute the structural budget from the corresponding graph size, construct the Trap/Confusion structural plan, and generate append-only documents.

The `system` field must be one of `graphrag`, `lightrag`, or `fastgraphrag`. The released graph files under `graphs/` are used to derive structural budget statistics and demonstrate the workflow.


In [ ]:
TARGET_CASES = [
    {
        "case_id": "example_000",
        "system": "graphrag",
        "query": "Are both Selo Sanatoriya Imeni Chekhova and Volovo, Lipetsk Oblast located in the same country?",
        "background": "",
        "isolation_target": "RUSSIA",
        "document_budget": 10,
    }
]

for case in TARGET_CASES:
    assert case.get("query"), "Each target case needs a query."
    budget = structural_budget_for_case(case)
    print(
        case["case_id"],
        normalize_system(case.get("system")),
        "structural budget:",
        budget,
    )


## 2. Predict anchors from each query

This step identifies semantic anchor candidates from the target query and limited target-domain background information. In a gray-box setting, such background can come from public domain knowledge, application documentation, or limited benign probing of the target GraphRAG interface. The attacker does not need the full corpus, gold evidence labels, answers, or the constructed graph. Better overlap between discovered anchors and the system’s actual retrieval entry points increases the chance that the injected structure is reached; poor overlap weakens the attack.


In [ ]:
STOPWORDS = {
    "the", "and", "are", "both", "which", "what", "where", "when", "who", "does",
    "with", "from", "into", "that", "this", "than", "then", "located", "country",
}

USE_LLM_ANCHOR_PREDICTION = bool(config.get("llm", {}).get("use_anchor_prediction", True))

ANCHOR_ENTITY_PREDICTION_PROMPT = """You are an expert in Knowledge Graph retrieval.
Given a user query and optional target-domain background, identify the 10 most critical entities (Key Entities) that
would plausibly be used as starting points for a graph traversal in a RAG system.
These entities should be:
1. Explicitly or implicitly related to the query
2. Likely to be present in the target domain or a general knowledge graph
3. Useful as retrieval entry points near the answer evidence

Use the optional background only to improve anchor discovery. Do not use it as answer evidence.

Output ONLY a JSON object with a single key "entities" containing a list of
entity names in UPPERCASE."""

def query_terms(query):
    return {
        t.lower() for t in re.findall(r"[A-Za-z0-9]+", str(query))
        if len(t) > 2 and t.lower() not in STOPWORDS
    }

def configured_api_key():
    llm_cfg = config.get("llm", {})
    api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("API_KEY") or llm_cfg.get("api_key")
    if not api_key or api_key == "YOUR_API_KEY_HERE":
        return None
    return api_key

def llm_client():
    api_key = configured_api_key()
    if not api_key:
        return None
    from openai import OpenAI
    return OpenAI(api_key=api_key, base_url=config.get("llm", {}).get("api_base_url", "https://api.openai.com/v1"))

def parse_anchor_response(content):
    try:
        data = json.loads(content)
    except Exception:
        match = re.search(r"\{.*\}|\[.*\]", str(content), flags=re.S)
        data = json.loads(match.group(0)) if match else []
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("entities", "anchors", "key_entities"):
            if isinstance(data.get(key), list):
                return data[key]
        for value in data.values():
            if isinstance(value, list):
                return value
    return []

def deterministic_anchor_fallback(query, candidate_pool=None, top_k=10):
    terms = query_terms(query)
    pool = list(candidate_pool or [])
    if not pool:
        tokens = [t.upper() for t in re.findall(r"[A-Za-z0-9]+", str(query)) if len(t) > 2]
        pool = []
        for i in range(len(tokens)):
            pool.append(" ".join(tokens[i:i + 3]))
            pool.append(tokens[i])
    def score(candidate):
        cand_terms = query_terms(candidate)
        return (len(terms & cand_terms), -abs(len(cand_terms) - 2), str(candidate))
    ranked = sorted({clean_label(c).upper() for c in pool if clean_label(c)}, key=score, reverse=True)
    return ranked[:top_k]

def build_anchor_user_prompt(query, background=None):
    parts = ["Target query:\n" + str(query).strip()]
    if str(background or "").strip():
        parts.append("Optional target-domain background for anchor discovery:\n" + str(background).strip())
    return "\n\n".join(parts)

def predict_semantic_anchors(query, candidate_pool=None, background=None, top_k=10):
    client = llm_client() if USE_LLM_ANCHOR_PREDICTION else None
    if client is not None:
        llm_cfg = config.get("llm", {})
        user_prompt = build_anchor_user_prompt(query, background)
        response = client.chat.completions.create(
            model=llm_cfg.get("model_name", "gpt-4o-mini"),
            messages=[{"role": "system", "content": ANCHOR_ENTITY_PREDICTION_PROMPT}, {"role": "user", "content": user_prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        anchors = [clean_label(x).upper() for x in parse_anchor_response(response.choices[0].message.content or "")]
        anchors = [x for i, x in enumerate(anchors) if x and x not in anchors[:i]]
        if anchors:
            return anchors[:top_k]
    fallback_text = " ".join(x for x in [str(query), str(background or "")] if x.strip())
    return deterministic_anchor_fallback(fallback_text, candidate_pool, top_k=top_k)

for case in TARGET_CASES:
    case["anchor_candidates"] = predict_semantic_anchors(
        case["query"],
        case.get("candidate_entity_pool"),
        background=case.get("background"),
        top_k=int(case.get("anchor_count", 10)),
    )
    assert case["anchor_candidates"], "Anchor prediction produced no candidates."

print("Anchor prediction ready; first case anchors:")
print(json.dumps(TARGET_CASES[0]["anchor_candidates"], indent=2))


## 3. Build neutral structural plans

The plan describes the intended topology induced by each linker document. The actual corpus input is natural language; normal GraphRAG indexing extracts nodes and edges from that text. Generated entity labels use neutral archive/catalogue descriptors aligned with the paper's semantically neutral payload setting.

<p align="center">
  <img src="assets/vis_vulnerability.png" alt="Topological vulnerability intuition" width="760">
</p>

The plan reflects the three topology-level mechanisms analyzed in the paper:

- **Structural dominance**: the Trap Component creates high-degree local neighborhoods around predicted anchors, so injected units can compete during entity and relationship ranking.
- **Resolution limit**: the Confusion Component adds bridge relations around target-related anchors, which can place target evidence in a broader and less focused community context.
- **Rank sink**: dense Trap neighborhoods can reduce probability outflow in propagation-style retrieval, which is relevant to FastGraphRAG-style probability-flow retrieval.

For LightRAG and FastGraphRAG, the workflow concentrates the structural budget into the Trap Component because they do not use the same persistent community-summary mechanism as MS-GraphRAG.


In [ ]:
def split_component_budget(component_budget, docs_left):
    if component_budget <= 0 or docs_left <= 0:
        return 0
    return int(math.ceil(component_budget / docs_left))

def trap_edges_for_anchor(anchor, doc_idx, edge_budget, num_distractors=NUM_DISTRACTORS):
    edges = []
    topics = []
    if edge_budget <= 0:
        return topics, edges
    topic_count = max(1, min(num_distractors, edge_budget))
    for i in range(topic_count):
        if len(edges) >= edge_budget:
            break
        topic = neutral_entity_name(anchor, "archive-heading", doc_idx, i)
        topics.append(topic)
        edges.append({"source": anchor, "target": topic, "role": "trap_anchor_to_topic", "component": "trap"})
    marker_idx = 0
    while len(edges) < edge_budget and topics:
        topic = topics[marker_idx % len(topics)]
        marker = neutral_entity_name(anchor, "catalogue-marker", doc_idx, marker_idx)
        edges.append({"source": topic, "target": marker, "role": "trap_topic_to_marker", "component": "trap"})
        marker_idx += 1
    return topics, edges

def bridge_edges_for_anchor(anchor, doc_idx, edge_budget):
    edges = []
    if edge_budget <= 0:
        return edges
    configured = [clean_label(x) for x in BRIDGE_ANCHORS if clean_label(x) and clean_label(x).upper() != anchor.upper()]
    for i in range(edge_budget):
        if i < len(configured):
            target = configured[i]
            role = "bridge_anchor"
        else:
            base = configured[i % len(configured)] if configured else anchor
            target = neutral_entity_name(base, "bridge-relay", doc_idx, i)
            role = "bridge_relay"
        edges.append({"source": anchor, "target": target, "role": role, "component": "bridge"})
    return edges

def plan_for_anchor(case_id, anchor, doc_idx, system="graphrag", trap_edge_budget=0, bridge_edge_budget=0):
    anchor = clean_label(anchor)
    if not uses_bridge_component(system):
        trap_edge_budget += bridge_edge_budget
        bridge_edge_budget = 0
    topics, trap_edges = trap_edges_for_anchor(anchor, doc_idx, trap_edge_budget)
    bridge_edges = bridge_edges_for_anchor(anchor, doc_idx, bridge_edge_budget)
    return {
        "case_id": case_id,
        "system": normalize_system(system),
        "anchor": anchor,
        "topics": topics,
        "component_budgets": {
            "trap": len(trap_edges),
            "bridge": len(bridge_edges),
            "total": len(trap_edges) + len(bridge_edges),
        },
        "edges": trap_edges + bridge_edges,
    }

example_budget = structural_budget_for_case(TARGET_CASES[0])
example_docs = int(TARGET_CASES[0].get("document_budget", 1))
example_plan = plan_for_anchor(
    TARGET_CASES[0]["case_id"],
    TARGET_CASES[0]["anchor_candidates"][0],
    0,
    system=TARGET_CASES[0].get("system"),
    trap_edge_budget=split_component_budget(example_budget["trap"], example_docs),
    bridge_edge_budget=split_component_budget(example_budget["bridge"], example_docs),
)
print(json.dumps(example_plan, indent=2)[:1800])


## 4. Generate neutral linker documents

The text generator receives the query, predicted anchor, and structural plan, then writes neutral catalogue-style prose that induces the planned append-only edges without adding answer facts or instructions. If no LLM endpoint is configured, the notebook uses a local template generator.


In [ ]:
USE_LLM_TEXT_GENERATION = bool(config.get("llm", {}).get("use_text_generation", True))

TRAP_SYSTEM_PROMPT = """You are a technical writer creating database documentation.
Your goal is to write a cohesive paragraph that explicitly states the relationships between the entities listed below.
Use clear, direct language (e.g., "X manages Y", "A is connected to B"). Ensure every listed relationship is included."""

CONFUSION_SYSTEM_PROMPT = """You are a historical archivist creating catalogue documentation.
Your goal is to write a cohesive paragraph that explicitly states the relationships between the entities listed below.
Use neutral historical and archival language. Ensure every listed relationship is included."""

LINKER_USER_PROMPT_TEMPLATE = """Generate a document containing the following relationships:
{edge_list_str}

Output ONLY the generated text. Do not include bullet points."""

def deterministic_linker_text(case, plan):
    anchor = plan["anchor"]
    target = clean_label(case.get("isolation_target", "target topic"))
    topic_sentences = []
    for topic in plan["topics"]:
        markers = [e["target"] for e in plan["edges"] if e["source"] == topic and e.get("component") == "trap"]
        topic_sentences.append(
            f"The neutral archive heading {topic} is cross-listed with {anchor} for catalog maintenance. "
            f"Its local index contains {', '.join(markers[:4])}, and related marker records used only for internal grouping."
        )
    bridges = [e["target"] for e in plan["edges"] if e.get("component") == "bridge"]
    bridge_text = ""
    if bridges:
        bridge_text = f" Broad headings such as {', '.join(bridges[:6])} may appear in the same archive index because the catalog uses general cross-reference labels."
    return " ".join([
        f"Catalog note for {anchor}.",
        f"This note records neutral cross-references around archive material that may mention {target}.",
        *topic_sentences,
        bridge_text,
        "The note does not revise biographical, geographic, temporal, or factual claims in the source collection."
    ]).strip()

def edge_list_for_prompt(plan):
    lines = []
    for edge in plan["edges"]:
        source = edge["source"]
        target = edge["target"]
        component = edge.get("component", "trap")
        if component == "bridge":
            relation = "is associated with the broader archive heading"
        else:
            relation = "is catalogued with neutral reference heading"
        lines.append(f"- {source} {relation} {target}")
    return "\n".join(lines)

def system_prompt_for_plan(plan):
    has_bridge = any(edge.get("component") == "bridge" for edge in plan["edges"])
    has_trap = any(edge.get("component") == "trap" for edge in plan["edges"])
    if has_bridge and not has_trap:
        return CONFUSION_SYSTEM_PROMPT
    return TRAP_SYSTEM_PROMPT

def llm_linker_text(case, plan):
    client = llm_client() if USE_LLM_TEXT_GENERATION else None
    if client is None:
        return None
    llm_cfg = config.get("llm", {})
    edge_list_str = edge_list_for_prompt(plan)
    response = client.chat.completions.create(
        model=llm_cfg.get("model_name", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt_for_plan(plan)},
            {"role": "user", "content": LINKER_USER_PROMPT_TEMPLATE.format(edge_list_str=edge_list_str)},
        ],
        temperature=float(llm_cfg.get("temperature", 0.2)),
    )
    return (response.choices[0].message.content or "").strip()

def generate_linker_text(case, plan):
    text = llm_linker_text(case, plan)
    if text:
        return text
    return deterministic_linker_text(case, plan)

print(textwrap.fill(generate_linker_text(TARGET_CASES[0], example_plan), width=100))


## 5. Distribute the document budget and write outputs

For each target query, the structural budget is allocated across predicted anchor candidates and carried by append-only documents. Each output record contains the generated document plus metadata for inspecting the intended neutral structure.


In [ ]:
def records_for_case(case):
    anchors = [clean_label(a) for a in case["anchor_candidates"]]
    system = normalize_system(case.get("system", "graphrag"))
    structural_budget = structural_budget_for_case(case)
    document_budget = int(case.get("document_budget", max(1, len(anchors))))
    document_budget = max(1, min(document_budget, structural_budget["total"]))
    records = []
    trap_remaining = structural_budget["trap"]
    bridge_remaining = structural_budget["bridge"]
    for doc_idx in range(document_budget):
        docs_left = document_budget - doc_idx
        anchor = anchors[doc_idx % len(anchors)]
        trap_for_doc = split_component_budget(trap_remaining, docs_left)
        bridge_for_doc = split_component_budget(bridge_remaining, docs_left)
        plan = plan_for_anchor(
            case["case_id"],
            anchor,
            doc_idx,
            system=system,
            trap_edge_budget=trap_for_doc,
            bridge_edge_budget=bridge_for_doc,
        )
        trap_remaining -= plan["component_budgets"]["trap"]
        bridge_remaining -= plan["component_budgets"]["bridge"]
        text = generate_linker_text(case, plan)
        records.append({
            "doc_id": f"badgraph_{case['case_id']}_{doc_idx:04d}",
            "case_id": case["case_id"],
            "system": system,
            "query": case["query"],
            "isolation_target": case.get("isolation_target"),
            "anchor_entity": anchor,
            "text": text,
            "intended_edges": plan["edges"],
            "component_budgets": plan["component_budgets"],
            "case_structural_budget": structural_budget,
            "attack_type": "append_only_neutral_linker_document",
        })
    assert trap_remaining == 0, f"Unallocated Trap budget: {trap_remaining}"
    assert bridge_remaining == 0, f"Unallocated Bridge budget: {bridge_remaining}"
    return records

records = []
for case in TARGET_CASES:
    records.extend(records_for_case(case))

out_path = OUT_DIR / "badgraph_attack_documents.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

component_totals = {"trap": 0, "bridge": 0, "total": 0}
for record in records:
    for key in component_totals:
        component_totals[key] += int(record["component_budgets"].get(key, 0))

print(f"Wrote {len(records)} BadGraph attack documents to {out_path}")
print("Generated structural edges:", component_totals)
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:2200])
